# Multi-Agent Coordination with Redis Streams

This notebook demonstrates `AgentCoordinator` for real-time coordination between distributed agent instances using Redis Streams.

## Use Cases

- **Handoff Notifications** - Notify target agents when handoffs are ready
- **Tool Result Broadcasting** - Share tool results with multiple consumers
- **State Synchronization** - Keep distributed replicas in sync
- **Crash Recovery** - Claim pending messages from failed consumers

In [ ]:
import os
import asyncio

from redis_openai_agents import AgentCoordinator, EventType

# Configuration
REDIS_URL = os.getenv("REDIS_URL", "redis://localhost:6379")

print(f"Redis URL: {REDIS_URL}")

## 1. Initialize Coordinators

Create a publisher and subscriber coordinator. In production, these would be separate processes.

In [ ]:
# Publisher coordinator (for publishing events)
publisher = AgentCoordinator(
    redis_url=REDIS_URL,
    stream_name="demo_agent_events",
)
await publisher.initialize()

# Subscriber coordinator (for consuming events)
subscriber = AgentCoordinator(
    redis_url=REDIS_URL,
    stream_name="demo_agent_events",
    consumer_group="demo_workers",
    consumer_name="worker_1",
)
await subscriber.initialize()

print("Publisher and subscriber initialized")

## 2. Event Types

The coordinator supports several standard event types for agent coordination.

In [ ]:
print("Available Event Types:")
print("=" * 40)
for event_type in EventType:
    print(f"  - {event_type.name}: {event_type.value}")

## 3. Publish Handoff Notification

When an agent hands off to another, it publishes a notification.

In [ ]:
# Publish handoff from "research_agent" to "analysis_agent"
msg_id = await publisher.publish_handoff_ready(
    from_agent="research_agent",
    to_agent="analysis_agent",
    session_id="session_demo_123",
    context={
        "topic": "Redis performance optimization",
        "findings": ["caching helps", "use pipelining"],
    },
)

print(f"Published handoff notification: {msg_id}")

## 4. Publish Tool Result

Broadcast tool completion results to all interested consumers.

In [ ]:
# Publish tool result
msg_id = await publisher.publish_tool_result(
    tool_name="web_search",
    session_id="session_demo_123",
    result={
        "urls": ["https://redis.io/docs", "https://github.com/redis/redis"],
        "summary": "Found Redis documentation and source code.",
    },
    execution_time_ms=245.5,
)

print(f"Published tool result: {msg_id}")

## 5. Publish Agent Lifecycle Events

Track when agents start and complete processing.

In [ ]:
# Agent started
await publisher.publish_agent_started(
    agent_name="analysis_agent",
    session_id="session_demo_123",
    input_summary="Analyze Redis performance findings",
)

# Simulate work
await asyncio.sleep(0.1)

# Agent completed
await publisher.publish_agent_completed(
    agent_name="analysis_agent",
    session_id="session_demo_123",
    output_summary="Analysis complete with 3 recommendations",
    duration_ms=150.0,
    tokens_used=500,
)

print("Published agent lifecycle events")

## 6. Subscribe to Events

Consume events from the stream with automatic acknowledgment.

In [ ]:
print("Subscribing to events (reading up to 10):")
print("=" * 60)

async for event in subscriber.subscribe(
    event_types=None,  # All event types
    timeout_ms=1000,
    max_events=10,
):
    event_type = event.get("type", "unknown")
    session_id = event.get("session_id", "N/A")
    
    print(f"\n[{event_type.upper()}] Session: {session_id}")
    
    if event_type == "handoff_ready":
        print(f"  From: {event.get('from_agent')} -> To: {event.get('to_agent')}")
    elif event_type == "tool_result":
        print(f"  Tool: {event.get('tool_name')} ({event.get('execution_time_ms')}ms)")
    elif event_type == "agent_started":
        print(f"  Agent: {event.get('agent_name')} started")
    elif event_type == "agent_completed":
        print(f"  Agent: {event.get('agent_name')} ({event.get('duration_ms')}ms)")

print("\nDone consuming events!")

## 7. Filtered Subscription

Subscribe to only specific event types.

In [ ]:
# Publish a mix of events
await publisher.publish_handoff_ready(
    from_agent="agent_a",
    to_agent="agent_b",
    session_id="session_filter_demo",
    context={},
)
await publisher.publish_error(
    session_id="session_filter_demo",
    error_type="ValidationError",
    error_message="Invalid input format",
)
await publisher.publish_state_changed(
    session_id="session_filter_demo",
    changes={"status": "completed"},
)

print("Published mix of events")

In [ ]:
# Subscribe only to errors
print("Subscribing only to ERROR events:")
print("=" * 40)

async for event in subscriber.subscribe(
    event_types=[EventType.ERROR.value],
    timeout_ms=1000,
    max_events=5,
):
    print(f"  Error: {event.get('error_type')}: {event.get('error_message')}")

print("\nFiltered subscription complete!")

## 8. Stream Information

Get statistics about the stream.

In [ ]:
info = await publisher.get_stream_info()

print("Stream Information:")
print("=" * 40)
print(f"  Length: {info.get('length', 0)} messages")
print(f"  Consumer Groups: {len(info.get('groups', []))}")

for group in info.get('groups', []):
    print(f"    - {group.get('name')}: {group.get('consumers')} consumers, {group.get('pending')} pending")

## 9. Crash Recovery

Claim messages from crashed consumers.

In [ ]:
# In production, you'd call this periodically
# to recover messages from crashed workers

claimed = await subscriber.claim_abandoned_messages(
    min_idle_ms=10000,  # Claim messages idle for 10+ seconds
)

print(f"Claimed {len(claimed)} abandoned messages")
for msg in claimed[:3]:  # Show first 3
    print(f"  - Type: {msg.get('type')}")

## 10. Stream Trimming

Keep the stream from growing unbounded.

In [ ]:
# Trim stream to keep only last 1000 messages
trimmed = await publisher.trim_stream(
    max_length=1000,
    approximate=True,  # More efficient
)

print(f"Trimmed {trimmed} messages from stream")

## Summary

The `AgentCoordinator` provides:

1. **Event Publishing** - Handoffs, tool results, lifecycle events, errors
2. **Event Subscription** - Filtered consumption with auto-acknowledgment
3. **Consumer Groups** - Work distribution across multiple consumers
4. **Crash Recovery** - Claim pending messages from failed consumers
5. **Stream Management** - Info and trimming

Use this for:
- Real-time handoff notifications (low latency vs polling)
- Tool result broadcasting to multiple agents
- Distributed agent coordination

In [ ]:
# Cleanup
await publisher.close()
await subscriber.close()
print("Connections closed.")